In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from glob import glob

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
MODEL_DIR = r"../models/baseline"

model = joblib.load(
    os.path.join(MODEL_DIR, "baseline_full_lightgbm.pkl")
)

feature_names = joblib.load(
    os.path.join(MODEL_DIR, "feature_names.pkl")
)

categorical_cols = joblib.load(
    os.path.join(MODEL_DIR, "categorical_cols.pkl")
)

feature_importance = pd.read_csv(
    os.path.join(MODEL_DIR, "feature_importance.csv")
)

with open(
    os.path.join(MODEL_DIR, "metrics.json"),
    "r"
) as f:
    baseline_metrics = json.load(f)

with open(
    os.path.join(MODEL_DIR, "model_params.json"),
    "r"
) as f:
    baseline_params = json.load(f)

In [3]:
print("Baseline Metrics:")
print(baseline_metrics)

print("\nNumber of Features:", len(feature_names))

print("\nCategorical Columns:")
print(categorical_cols)

print("\nTop 10 Features:")
print(feature_importance.head(10))

Baseline Metrics:
{'RMSE': 1.959555232531138, 'MAE': 0.9612995762426987}

Number of Features: 34

Categorical Columns:
['item_id', 'dept_id', 'cat_id', 'store_id', 'weekday', 'event_name_1', 'event_type_1']

Top 10 Features:
           feature  importance
0          item_id        2720
1            lag_1        2204
2   rolling_mean_7        1414
3         wm_yr_wk        1135
4            lag_7         969
5           lag_28         953
6           lag_14         831
7          weekday         788
8  rolling_mean_14         762
9         store_id         742


In [4]:
FEATURE_PATH = r"../data/processed/feature_chunks"

files = sorted(
    glob(
        os.path.join(
            FEATURE_PATH,
            "*.parquet"
        )
    )
)

selected_chunks = [0, 2, 5, 8]

selected_files = [
    files[i]
    for i in selected_chunks
]

for f in selected_files:
    print(os.path.basename(f))

features_chunk_0001.parquet
features_chunk_0003.parquet
features_chunk_0006.parquet
features_chunk_0009.parquet


In [5]:
train_parts = []
valid_parts = []

for file in selected_files:

    print(f"Loading {os.path.basename(file)}")

    df = pd.read_parquet(file)

    # نفس التقسيم المستخدم في Day 6
    split_idx = int(len(df) * 0.8)

    train_chunk = df.iloc[:split_idx]
    valid_chunk = df.iloc[split_idx:]

    train_parts.append(train_chunk)
    valid_parts.append(valid_chunk)

train_df = pd.concat(train_parts, ignore_index=True)
valid_df = pd.concat(valid_parts, ignore_index=True)

print("Train Shape:", train_df.shape)
print("Valid Shape:", valid_df.shape)

Loading features_chunk_0001.parquet
Loading features_chunk_0003.parquet
Loading features_chunk_0006.parquet
Loading features_chunk_0009.parquet
Train Shape: (18096000, 38)
Valid Shape: (4524000, 38)


In [6]:
drop_cols = [
    'sales',   # target
    'id',      # identifier
    'day',     # string day identifier
    'date'     # datetime column
]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['sales']

X_valid = valid_df.drop(columns=drop_cols)
y_valid = valid_df['sales']

In [7]:
for col in categorical_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_valid[col] = X_valid[col].astype("category")

In [8]:
X_train['state_id'] = X_train['state_id'].astype('category')
X_valid['state_id'] = X_valid['state_id'].astype('category')

categorical_cols.append('state_id')

In [11]:
import joblib

# Save validation features
joblib.dump(
    X_valid,
    r"../models/tuned/X_valid.pkl"
)

# Save validation target
joblib.dump(
    y_valid,
    r"../models/tuned/y_valid.pkl"
)

print("Validation data saved successfully.")

Validation data saved successfully.


In [9]:
from lightgbm import LGBMRegressor

model_v2 = LGBMRegressor(
    objective='regression',
    metric='rmse',
    learning_rate=0.03,
    n_estimators=500,
    num_leaves=128,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [10]:
X_train.dtypes.value_counts()

float32     16
int8         6
int16        2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

In [28]:
model_v2.fit(
    X_train,
    y_train,
    categorical_feature=categorical_cols
)

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, metric='rmse',
              n_estimators=500, n_jobs=-1, num_leaves=128,
              objective='regression', random_state=42, subsample=0.8,
              verbose=-1)

In [29]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

preds = model_v2.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, preds))
mae = mean_absolute_error(y_valid, preds)

print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")

RMSE: 1.2600
MAE : 0.4873


In [30]:
joblib.dump(
    model_v2,
    "../models/tuned/tuned_model_v1.pkl"
)

['tuned_model_v1.pkl']

In [31]:
tuned_metrics = {
    "RMSE": rmse,
    "MAE": mae
}

In [32]:
model_v3 = LGBMRegressor(
    objective='regression',
    metric='rmse',
    learning_rate=0.01,
    n_estimators=1000,
    num_leaves=128,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [33]:
results = pd.DataFrame({
    "Model": ["Baseline Full", "Tuned V1"],
    "RMSE": [1.9596, 1.2600],
    "MAE": [0.9613, 0.4873]
})

results

,Model,RMSE,MAE
0,Baseline Full,1.9596,0.9613
1,Tuned V1,1.2600,0.4873


In [34]:
model_v3 = LGBMRegressor(
    objective='regression',
    metric='rmse',
    learning_rate=0.01,
    n_estimators=1000,
    num_leaves=128,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [35]:
model_v3.fit(
    X_train,
    y_train,
    categorical_feature=categorical_cols
)

preds = model_v3.predict(X_valid)

rmse_v3 = np.sqrt(mean_squared_error(y_valid, preds))
mae_v3 = mean_absolute_error(y_valid, preds)

print(f"RMSE: {rmse_v3:.4f}")
print(f"MAE : {mae_v3:.4f}")


RMSE: 1.2603
MAE : 0.4885


In [36]:
model_v4 = LGBMRegressor(
    objective='regression',
    metric='rmse',
    learning_rate=0.03,
    n_estimators=500,
    num_leaves=64,
    colsample_bytree=0.8,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [37]:
model_v4.fit(
    X_train,
    y_train,
    categorical_feature=categorical_cols
)

preds = model_v4.predict(X_valid)

rmse_v4 = np.sqrt(mean_squared_error(y_valid, preds))
mae_v4 = mean_absolute_error(y_valid, preds)

print(f"RMSE: {rmse_v4:.4f}")
print(f"MAE : {mae_v4:.4f}")

RMSE: 1.2597
MAE : 0.4873


In [38]:
model_v5 = LGBMRegressor(
    objective='regression',
    metric='rmse',
    learning_rate=0.03,
    n_estimators=500,
    num_leaves=64,
    colsample_bytree=0.7,
    subsample=0.7,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [39]:
model_v5.fit(
    X_train,
    y_train,
    categorical_feature=categorical_cols
)

preds = model_v5.predict(X_valid)

rmse_v5 = np.sqrt(mean_squared_error(y_valid, preds))
mae_v5 = mean_absolute_error(y_valid, preds)

print(f"RMSE: {rmse_v5:.4f}")
print(f"MAE : {mae_v5:.4f}")

RMSE: 1.2605
MAE : 0.4876


In [43]:
import pandas as pd

results = []

for var_name, value in globals().items():

    if var_name.startswith("rmse"):
        
        suffix = var_name.replace("rmse", "")

        mae_name = f"mae{suffix}"

        if mae_name in globals():

            model_name = "Baseline" if suffix == "" else f"Model{suffix}"

            results.append({
                "Model": model_name,
                "RMSE": value,
                "MAE": globals()[mae_name]
            })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values("RMSE")

print(results_df.to_string(index=False))

   Model     RMSE      MAE
Model_v4 1.259666 0.487292
Baseline 1.259983 0.487257
Model_v3 1.260299 0.488491
Model_v5 1.260544 0.487563


In [41]:
print([x for x in globals().keys() if "rmse" in x.lower()])
print([x for x in globals().keys() if "mae" in x.lower()])

['rmse', 'rmse_v3', 'rmse_v4', 'rmse_v5']
['mae', 'mae_v3', 'mae_v4', 'mae_v5']


In [45]:
import pandas as pd

results = pd.DataFrame({
    "Version": [
        "Baseline",
        "Version 2",
        "Version 3",
        "Version 4",
        "Version 5"
    ],

    "RMSE": [
        metrics["RMSE"],   # baseline
        rmse_v2,
        rmse_v3,
        rmse_v4,
        rmse_v5
    ],

    "MAE": [
        metrics["MAE"],    # baseline
        mae_v2,
        mae_v3,
        mae_v4,
        mae_v5
    ]
})

results = results.sort_values("RMSE")

print(results.to_string(index=False))

NameError: name 'rmse_v2' is not defined

In [48]:
import pandas as pd

results = pd.DataFrame({
    "Version": [
        "Baseline",
        "Version 2",
        "Version 3",
        "Version 4",
        "Version 5"
    ],

    "RMSE": [
        metrics["RMSE"],
        rmse,
        rmse_v3,
        rmse_v4,
        rmse_v5
    ],

    "MAE": [
        metrics["MAE"],
        mae,
        mae_v3,
        mae_v4,
        mae_v5
    ]
})

results = results.sort_values("RMSE")

print(results.to_string(index=False))

  Version     RMSE      MAE
Version 4 1.259666 0.487292
Version 2 1.259983 0.487257
Version 3 1.260299 0.488491
Version 5 1.260544 0.487563
 Baseline 1.959555 0.961300


In [49]:
import os
import joblib
import json

SAVE_DIR = r"../models/tuned"

os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(
    model_v4,
    os.path.join(SAVE_DIR, "best_lightgbm_model.pkl")
)

['D:\\gearofproml\\day7_models\\best_lightgbm_model.pkl']

In [50]:
best_metrics = {
    "RMSE": rmse_v4,
    "MAE": mae_v4
}

with open(
    os.path.join(SAVE_DIR, "best_metrics.json"),
    "w"
) as f:
    json.dump(
        best_metrics,
        f,
        indent=4
    )

In [51]:
best_metrics = {
    "RMSE": rmse_v4,
    "MAE": mae_v4
}

with open(
    os.path.join(SAVE_DIR, "best_metrics.json"),
    "w"
) as f:
    json.dump(
        best_metrics,
        f,
        indent=4
    )

In [52]:
joblib.dump(
    feature_names,
    os.path.join(SAVE_DIR, "feature_names.pkl")
)

joblib.dump(
    categorical_cols,
    os.path.join(SAVE_DIR, "categorical_cols.pkl")
)

['D:\\gearofproml\\day7_models\\categorical_cols.pkl']